# English → Bhojpuri Neural Machine Translation (NMT)
### 10-Hour Hackathon Execution Plan — Member 1 (ML / NLP Workflow)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook covers:
- **Step 4: Tokenizer & Model Setup** (`facebook/nllb-200-distilled-600M`)
- **Step 5: Baseline Check** (tokenization, dynamic padding, loss computation, greedy decoding)
- **Step 6: Fine-Tuning** (GPU training using Hugging Face `Seq2SeqTrainer`)
- **Step 7: Inference Function** (`translate(text) -> str`)
- **Step 8: BLEU Evaluation** on held-out test set

## 1. Environment & GPU Verification
In Google Colab, enable GPU acceleration:
> **Runtime** -> **Change runtime type** -> **T4 GPU** -> **Save**

In [ ]:
# Install required libraries in Google Colab
!pip install -q "transformers>=4.38.0" "datasets>=2.18.0" "evaluate>=0.4.1" "sacrebleu>=2.4.0" "sentencepiece>=0.2.0" "accelerate>=0.28.0"

import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Load Dataset Splits
Load our reproducible train, validation, and held-out test sets.

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Check if dataset files exist locally in Colab runtime
train_path = "data/train_2k.csv" if os.path.exists("data/train_2k.csv") else "train_2k.csv"
val_path = "data/val.csv" if os.path.exists("data/val.csv") else "val.csv"
test_path = "data/test.csv" if os.path.exists("data/test.csv") else "test.csv"

if os.path.exists(train_path) and os.path.exists(val_path):
    print(f"Loading local split files: {train_path}, {val_path}...")
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path) if os.path.exists(test_path) else val_df
else:
    print("Splits not found locally. Pulling and splitting directly from Hugging Face dataset...")
    from datasets import load_dataset
    import re
    
    hf_data = load_dataset("nilayshenai/English-Bhojpuri_Translation_Dataset", split="train")
    pairs = []
    for x in hf_data:
        t = x.get('translation', {})
        en = str(t.get('en', '')).strip()
        bho = str(t.get('bho', '')).strip()
        if en and bho and re.search(r'[a-zA-Z]', en) and re.search(r'[\u0900-\u097F]', bho):
            pairs.append({'en': en, 'bho': bho})
            
    full_df = pd.DataFrame(pairs).drop_duplicates(subset=['en', 'bho']).sample(frac=1, random_state=42).reset_index(drop=True)
    # Fast hackathon configuration: 2,000 train (Step 14 benchmark), 500 val, 500 test
    train_df = full_df.iloc[:2000].reset_index(drop=True)
    val_df = full_df.iloc[2000:2500].reset_index(drop=True)
    test_df = full_df.iloc[2500:3000].reset_index(drop=True)

print(f"Dataset ready: Train={len(train_df):,} pairs | Val={len(val_df):,} pairs | Test={len(test_df):,} pairs")
raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df)
})
print(raw_datasets)

## 3. Step 4 — Tokenizer & SentencePiece Subword Tokenization
We load Meta's `facebook/nllb-200-distilled-600M` tokenizer:
- **Source Language**: `eng_Latn` (English in Latin script)
- **Target Language**: `bho_Deva` (Bhojpuri in Devanagari script)

### Subword/BPE Mechanism:
SentencePiece decomposes infrequent and dialectal Bhojpuri words into frequent shared subword units and byte fallbacks, completely preventing Out-Of-Vocabulary (`<unk>`) errors.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
TGT_LANG = "bho_Deva"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG,
    use_fast=True
)

# Live subword inspection
sample_en = "The foreman says you have to work tonight!"
sample_bho = "परबंधक क कहल हव की आपको काम करय के हव।"

tokenizer.src_lang = SRC_LANG
print("English Subwords:", tokenizer.tokenize(sample_en))
tokenizer.src_lang = TGT_LANG
print("Bhojpuri Subwords:", tokenizer.tokenize(sample_bho))
tokenizer.src_lang = SRC_LANG

## 4. Tokenization & Data Collator

In [ ]:
from transformers import DataCollatorForSeq2Seq

def preprocess_function(examples):
    inputs = [str(x) for x in examples['en']]
    targets = [str(x) for x in examples['bho']]
    
    tokenizer.src_lang = SRC_LANG
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    
    tokenizer.src_lang = TGT_LANG
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    tokenizer.src_lang = SRC_LANG
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True, remove_columns=['en', 'bho'])
print("Tokenized features:", tokenized_datasets['train'].column_names)

## 5. Step 5 — Baseline Check (Model Loading & Sanity Check)
Load the pretrained weights in FP16 and verify the forward pass and loss computation on a sample batch.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

print(f"Loading pretrained Seq2Seq model: {MODEL_NAME}...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

# Sanity check forward pass on 1 sample batch
dummy_batch = data_collator([tokenized_datasets['train'][0]])
dummy_batch = {k: v.to(device) for k, v in dummy_batch.items()}

with torch.no_grad():
    out = model(**dummy_batch)

print(f"Baseline Loss Check: {out.loss.item():.4f}")
print("Baseline sanity test passed! Ready for training.")

## 6. Step 6 — Fine-Tuning Execution (`Seq2SeqTrainer`)
Fine-tune the model with FP16 mixed precision and gradient accumulation.

In [ ]:
import inspect
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

checkpoint_dir = "./models/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Support both newer and older transformers syntax for evaluation strategy
strategy_key = "eval_strategy" if hasattr(Seq2SeqTrainingArguments(output_dir="tmp"), "eval_strategy") else "evaluation_strategy"

args_dict = {
    "output_dir": checkpoint_dir,
    strategy_key: "epoch",
    "save_strategy": "epoch",
    "learning_rate": 3e-5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "weight_decay": 0.01,
    "save_total_limit": 1,
    "num_train_epochs": 3,
    "predict_with_generate": True,
    "fp16": torch.cuda.is_available(),
    "logging_steps": 25,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "report_to": "none"
}

training_args = Seq2SeqTrainingArguments(**args_dict)

# Build trainer arguments dynamically to support both transformers v4 ('tokenizer') and v5 ('processing_class')
trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator
}

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("Starting Seq2Seq fine-tuning on GPU...")
trainer.train()

# Save best model and tokenizer
best_model_dir = "./models/bhojpuri-nmt-best"
os.makedirs(best_model_dir, exist_ok=True)
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
print(f"Best model saved successfully to: {best_model_dir}")

## 7. Step 7 — Inference Function
Standalone `translate(text) -> str` function to test 10 unseen sentences and hand over to Member 2 for Gradio.

In [ ]:
target_lang_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

def translate(text: str, max_length: int = 128) -> str:
    """Translate an English sentence into Bhojpuri."""
    model.eval()
    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    
    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_new_tokens=max_length,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

# Test 10 unseen English sentences
test_samples = [
    "Hello, how are you?",
    "Where are you going today?",
    "Since childhood, she has never asked me anything till now.",
    "The foreman says you have to work tonight!",
    "May I sit on my bench?",
    "Indeed he deserved to live.",
    "What do you mean, he just wasn't here?",
    "I want to learn Bhojpuri language.",
    "We built this translator during the hackathon.",
    "Good morning, my friend!"
]

print("\n--- Sample Translations ---")
for s in test_samples:
    print(f"EN:  {s}")
    print(f"BHO: {translate(s)}")
    print("-" * 50)

## 8. Step 8 — BLEU Evaluation on Held-Out Test Set
Calculates corpus BLEU using SacreBLEU and outputs `results/bleu.txt` and `results/sample_translations.csv`.

In [ ]:
import evaluate
from tqdm.auto import tqdm

sacrebleu = evaluate.load("sacrebleu")

# Evaluate on 200 held-out test sentences
eval_subset = test_df.head(200)
references = [[str(ref).strip()] for ref in eval_subset['bho']]
predictions = []

print(f"Generating translations for {len(eval_subset)} test sentences...")
for src_text in tqdm(eval_subset['en']):
    predictions.append(translate(str(src_text).strip()))

results = sacrebleu.compute(predictions=predictions, references=references)
corpus_bleu = results['score']
print(f"\nCorpus BLEU Score on held-out test set: {corpus_bleu:.2f}")

# Save evaluation results
os.makedirs("results", exist_ok=True)
with open("results/bleu.txt", "w", encoding="utf-8") as f:
    f.write(f"Corpus BLEU: {corpus_bleu:.2f}\nTest Sentences: {len(eval_subset)}\nModel: {MODEL_NAME}\n")

eval_df = pd.DataFrame({
    "English": eval_subset['en'],
    "Reference_Bhojpuri": eval_subset['bho'],
    "Predicted_Bhojpuri": predictions
})
eval_df.to_csv("results/sample_translations.csv", index=False, encoding="utf-8")
print("Results saved to results/bleu.txt and results/sample_translations.csv")

# Download trained model and results zip file to your local machine
# from google.colab import files
# !zip -r bhojpuri-nmt-best.zip ./models/bhojpuri-nmt-best results/
# files.download('bhojpuri-nmt-best.zip')